# 03 — Evaluation
Statistical and economic evaluation of all models. Load walk-forward results and compute IC, portfolio metrics, and cumulative returns.

In [ ]:
import sys, warnings, pickle
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

plt.rcParams.update({'figure.dpi': 130, 'axes.spines.top': False,
                     'axes.spines.right': False, 'font.size': 11})

with open('../results/walk_forward_results.pkl', 'rb') as f:
    saved = pickle.load(f)

results      = saved['results']
feature_names = saved['feature_names']
print("Models loaded:", list(results.keys()))


## Statistical metrics

In [ ]:
from src.evaluation import statistical_summary

stats = statistical_summary(results)
print(stats.to_string())
stats.to_csv('../results/statistical_metrics.csv')


## IC time series

In [ ]:
from src.evaluation import compute_ic_series

COLOURS = {
    'OLS (FF5+Mom+STRev)': '#2166ac',
    'Lasso':               '#4dac26',
    'Random Forest':       '#d01c8b',
    'XGBoost':             '#f1a340',
}

fig, ax = plt.subplots(figsize=(12, 4))
for name, res in results.items():
    ic = compute_ic_series(res['predictions'], res['actuals'])
    ic_roll = ic.rolling(12, min_periods=6).mean()
    ax.plot(ic_roll.index, ic_roll.values, label=name,
            color=COLOURS.get(name), linewidth=1.8)

ax.axhline(0, color='black', linewidth=0.8, linestyle='--', alpha=0.5)
ax.set_title('12-month rolling information coefficient (IC) by model', fontsize=12)
ax.set_ylabel('IC (Spearman rank correlation)')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout()
plt.savefig('../results/figures/ic_over_time.png', bbox_inches='tight')
plt.show()


## Portfolio construction and metrics

In [ ]:
from src.evaluation import build_long_short_portfolio, portfolio_summary

port_returns   = {}
port_summaries = {}

for name, res in results.items():
    port = build_long_short_portfolio(res['predictions'], res['raw_returns'])
    port_returns[name]   = port
    port_summaries[name] = portfolio_summary(port)
    for leg in ['Gross', 'Net']:
        s = port_summaries[name][leg]
        print(f"{name} [{leg}]  Sharpe={s['Sharpe']:+.3f}  "
              f"Ret={s['Ann. Return (%)']:+.1f}%  "
              f"Vol={s['Ann. Vol (%)']:.1f}%  MaxDD={s['Max DD (%)']:.1f}%")
    print()


## Cumulative returns

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 7), sharex=True)
axes = axes.flatten()

for i, (name, port) in enumerate(port_returns.items()):
    ax = axes[i]
    gross_cum = (1 + port['Gross']).cumprod()
    net_cum   = (1 + port['Net']).cumprod()
    ax.plot(gross_cum.index, gross_cum.values, label='Gross',
            color=COLOURS.get(name, 'steelblue'), linewidth=1.8)
    ax.plot(net_cum.index, net_cum.values, label='Net (−10bps)',
            color=COLOURS.get(name, 'steelblue'), linewidth=1.8, linestyle='--')
    ax.axhline(1, color='black', linewidth=0.5, linestyle=':', alpha=0.4)
    ax.set_title(name, fontsize=11)
    ax.legend(fontsize=8)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

fig.suptitle('Cumulative long-short portfolio returns (top vs bottom tertile)', fontsize=12)
plt.tight_layout()
plt.savefig('../results/figures/cumulative_returns.png', bbox_inches='tight')
plt.show()


## Sharpe comparison: gross vs net

In [ ]:
from src.evaluation import plot_sharpe_comparison
plot_sharpe_comparison(port_summaries)

# Summary table
rows = []
for name, sums in port_summaries.items():
    for leg, m in sums.items():
        rows.append({'Model': name, 'Leg': leg, **m})
port_df = pd.DataFrame(rows)
port_df.to_csv('../results/portfolio_metrics.csv', index=False)
port_df
